In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import statsmodels.api as sm
import os

import refinitiv.dataplatform as rdp

data_folder = os.getcwd() + '/'
data_folder
# https://github.com/LSEG-API-Samples/Example.RDPLibrary.Python

'/Users/gunassavsopee/Gun/Python/TEST_MVP/MACRO_DATA/'

In [11]:
def cleandata(df):
    df = df.astype(float)
    # df[col_name]= np.log(df[col_name]/df[col_name].shift(1))
    df = df.asfreq('d').ffill()  # Fill missing values
    df = df.dropna()
    return df

In [12]:
country = {'EZU':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'EUR/USD':['EUR=','MID_PRICE',1],'GBY2Y':['EU2YT=RR','B_YLD_1',0],'GBY5Y':['EU5YT=RR','B_YLD_1',0],'GBY10Y':['EU10YT=RR','B_YLD_1',0]},
           
           'EWA':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'EUR/USD':['AUD=','MID_PRICE',1],'GBY2Y':['AU2YT=RR','B_YLD_1',0],'GBY5Y':['AU5YT=RR','B_YLD_1',0],'GBY10Y':['AU10YT=RR','B_YLD_1',0]},
           
           'EWJ':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'JPY/USD':['JPY=','MID_PRICE',1],'GBY2Y':['JP2YT=RR','B_YLD_1',0],'GBY5Y':['JP5YT=RR','B_YLD_1',0],'GBY10Y':['JP10YT=RR','B_YLD_1',0]},
           
           'EWU':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'GBP/USD':['GBP=','MID_PRICE',1],'GBY2Y':['GB2YT=RR','B_YLD_1',0],'GBY5Y':['GB5YT=RR','B_YLD_1',0],'GBY10Y':['GB10YT=RR','B_YLD_1',0]},
           
           'IVV':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'USD':['=USD','MID_PRICE',1],'GBY2Y':['US2YT=RR','B_YLD_1',0],'GBY5Y':['US5YT=RR','B_YLD_1',0],'GBY10Y':['US10YT=RR','B_YLD_1',0]},
           
           'MCHI':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                   'CNY/USD':['CNY=','MID_PRICE',1],'GBY2Y':['CN2YT=RR','B_YLD_1',0],'GBY5Y':['CN5YT=RR','B_YLD_1',0],'GBY10Y':['CN10YT=RR','B_YLD_1',0]},
           
           'INDA':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                   'INR/USD':['INR=','MID_PRICE',1],'GBY2Y':['IN2YT=RR','B_YLD_1',0],'GBY5Y':['IN5YT=RR','B_YLD_1',0],'GBY10Y':['IN10YT=RR','B_YLD_1',0]},
           
           'VNM':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'VND/USD':['VND=','MID_PRICE',1],'GBY2Y':['VN2YT=RR','B_YLD_1',0],'GBY5Y':['VN5YT=RR','B_YLD_1',0],'GBY10Y':['VN10YT=RR','B_YLD_1',0]},
           
           'THD':{'gold':['XAU=','MID_PRICE',1],'BDI':['.BADI','TRDPRC_1',1],'BRT':['BRT-','MID_PRICE',1],'VIX':['.VIX','TRDPRC_1',0],
                  'THB/USD':['THB=','MID_PRICE',1],'GBY2Y':['TH2YT=RR','B_YLD_1',0],'GBY5Y':['TH5YT=RR','B_YLD_1',0],'GBY10Y':['TH10YT=RR','B_YLD_1',0]}}


In [13]:
rdp.open_platform_session("cee1bdf067914b2fb38c11b3fa8c90025968b919",
    rdp.GrantPassword("6884008226@cbs.chula.ac.th","Paipai@00"
    )
)
start_date = '2010-01-01'
end_date = '2025-12-31'
for i in country :
    data = pd.DataFrame()
    for j in country[i]:
        # print(i,j,country[i][j])
        df = rdp.get_historical_price_summaries(universe = country[i][j][0] ,interval=rdp.Intervals.DAILY,fields = [country[i][j][1]],start=start_date,end=end_date)
        df = df.rename(columns={country[i][j][1]: j})
        df = cleandata(df)
        # if country[i][j][2] == 1 :
        #     df[j+'_%diff_20'] = ((df[j]/df[j].shift(20))-1)*100
        #     df.drop(j, axis=1, inplace=True)
        data = data.merge(df,left_index=True,right_index=True,how='outer')
        
    data = cleandata(data)
    data['GBY10-2Y'] =data['GBY10Y']-data['GBY2Y']
    
    file_name = i + '_RAW_Macro_D'
    data.to_excel(file_name + '.xlsx')
    print(file_name+' DONE')

EZU_Macro_D DONE
EWA_Macro_D DONE
EWJ_Macro_D DONE
EWU_Macro_D DONE
IVV_Macro_D DONE
MCHI_Macro_D DONE
INDA_Macro_D DONE
VNM_Macro_D DONE
THD_Macro_D DONE
